# Extracción MongoDB → Excel (schema completo normalizado)
**Filtro:** `cola_id = 43` | Fecha: hoy automático

Todas las columnas del schema `prueba.json` aparecen en el DataFrame aunque estén vacías.
- Objetos anidados (`contacto`, `marcador_info`) → columnas con prefijo
- Arrays (`gestiones`, `gestiones_old`, `detalle`) → una fila por ítem expandido (join con datos raíz)

In [1]:
# pip install pymongo pandas openpyxl dnspython
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

pd.set_option('display.max_columns', None)

In [2]:
# ── CONFIGURACIÓN ─────────────────────────────────────────────────────────────
host_mongo = "44.195.114.98"
user_mongo = "bi_guest"
pwd_mongo = "gu3$t202X#"
db_name_mongo = "reportes"

MONGO_URI        = f"mongodb://{user_mongo}:{pwd_mongo}@{host_mongo}:27017/{db_name_mongo}"  # <-- tu URI
DATABASE         = "reportes"
COLLECTION       = "log_comunicaciones"
COLA_ID          = 43
SCHEMA_JSON_PATH = Path(r"C:\Users\USUARIO\Downloads\NTP_CODEX\prueba.json")
OUTPUT_EXCEL     = Path(r"C:\Users\USUARIO\Downloads\NTP_CODEX\extraccion_hoy.xlsx")

In [3]:
# ── RANGO DE FECHA (hoy, hora local Perú UTC-5) ───────────────────────────────
from datetime import timedelta

ahora      = datetime.now(timezone.utc)
hoy_inicio = ahora.replace(hour=5, minute=0, second=0, microsecond=0)          # 00:00 Perú = 05:00 UTC
hoy_fin    = hoy_inicio + timedelta(hours=23, minutes=59, seconds=59)

# Si ya pasó medianoche en Perú pero aún es el día anterior en UTC, ajusta:
if ahora < hoy_inicio:
    hoy_inicio -= timedelta(days=1)
    hoy_fin    -= timedelta(days=1)

print(f"Rango UTC  : {hoy_inicio}  →  {hoy_fin}")
print(f"Fecha Perú : {(hoy_inicio - timedelta(hours=5)).strftime('%Y-%m-%d')}")

Rango UTC  : 2026-04-08 05:00:00+00:00  →  2026-04-09 04:59:59+00:00
Fecha Perú : 2026-04-08


In [4]:
# ── LEER SCHEMA Y CONSTRUIR LISTA DE COLUMNAS ─────────────────────────────────
with open(SCHEMA_JSON_PATH, encoding='utf-8') as f:
    schema = json.load(f)

props = schema["collections"]["reportes.log_comunicaciones"]["jsonSchema"]["properties"]

ARRAY_KEYS  = {"detalle", "gestiones", "gestiones_old"}
OBJECT_KEYS = {"contacto", "marcador_info"}

# ── Columnas raíz
root_cols = [k for k in props if k not in ARRAY_KEYS and k not in OBJECT_KEYS]

# ── Columnas contacto (incluyendo _id anidado)
contacto_props = props["contacto"]["properties"]
contacto_cols  = ["contacto._id.date", "contacto._id.timestamp"] + [
    f"contacto.{k}" for k in contacto_props if k != "_id"
]

# ── Columnas marcador_info
marcador_cols = [f"marcador_info.{k}" for k in props["marcador_info"]["properties"]]

# ── Columnas gestiones (items del array)
gestion_item_props   = props["gestiones"]["items"]["properties"]
gestiones_cols       = [f"gestiones.{k}" for k in gestion_item_props]

# ── Columnas gestiones_old
gestion_old_props    = props["gestiones_old"]["items"]["properties"]
gestiones_old_cols   = [f"gestiones_old.{k}" for k in gestion_old_props]

# ── Columnas detalle
detalle_item_props   = props["detalle"]["items"]["properties"]
detalle_cols         = [f"detalle.{k}" for k in detalle_item_props]

# ── Orden final de columnas
ALL_COLS = (
    root_cols +
    contacto_cols +
    marcador_cols +
    gestiones_cols +
    gestiones_old_cols +
    detalle_cols
)

print(f"Total columnas schema  : {len(ALL_COLS)}")
print(f"  Raíz                 : {len(root_cols)}")
print(f"  contacto             : {len(contacto_cols)}")
print(f"  marcador_info        : {len(marcador_cols)}")
print(f"  gestiones            : {len(gestiones_cols)}")
print(f"  gestiones_old        : {len(gestiones_old_cols)}")
print(f"  detalle              : {len(detalle_cols)}")

Total columnas schema  : 204
  Raíz                 : 95
  contacto             : 30
  marcador_info        : 35
  gestiones            : 17
  gestiones_old        : 17
  detalle              : 10


In [5]:
# ── CONEXIÓN Y EXTRACCIÓN ─────────────────────────────────────────────────────
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=8000)

try:
    client.admin.command('ping')
    print("Conexión exitosa")
except Exception as e:
    print(f"Error: {e}")
    raise

col = client[DATABASE][COLLECTION]

filtro = {
    "cola_id": COLA_ID,
    "fecha_ingreso_cola": {"$gte": hoy_inicio, "$lte": hoy_fin}
}

docs = list(col.find(filtro))
print(f"Documentos encontrados: {len(docs)}")

Conexión exitosa
Documentos encontrados: 2787


In [6]:
# ── FUNCIÓN DE NORMALIZACIÓN TOTAL ───────────────────────────────────────────
def safe(val):
    """Convierte ObjectId y tipos no serializables a string."""
    if isinstance(val, ObjectId):
        return str(val)
    if isinstance(val, datetime):
        return val.strftime('%Y-%m-%d %H:%M:%S')
    if isinstance(val, (list, dict)):
        return json.dumps(val, default=str, ensure_ascii=False)
    return val


def flatten_doc(doc: dict) -> dict:
    flat = {}

    # ── Raíz
    for k in root_cols:
        flat[k] = safe(doc.get(k))

    # ── contacto
    contacto = doc.get("contacto") or {}
    cid = contacto.get("_id") or {}
    flat["contacto._id.date"]      = cid.get("date")      if isinstance(cid, dict) else None
    flat["contacto._id.timestamp"] = cid.get("timestamp") if isinstance(cid, dict) else None
    for k in contacto_props:
        if k != "_id":
            flat[f"contacto.{k}"] = safe(contacto.get(k))

    # ── marcador_info
    marcador = doc.get("marcador_info") or {}
    for k in props["marcador_info"]["properties"]:
        flat[f"marcador_info.{k}"] = safe(marcador.get(k)) if isinstance(marcador, dict) else None

    # ── gestiones (última gestión del array)
    gestiones = doc.get("gestiones") or []
    g = gestiones[-1] if gestiones else {}
    for k in gestion_item_props:
        flat[f"gestiones.{k}"] = safe(g.get(k)) if isinstance(g, dict) else None

    # ── gestiones_old (última)
    g_old = (doc.get("gestiones_old") or [])
    go = g_old[-1] if g_old else {}
    for k in gestion_old_props:
        flat[f"gestiones_old.{k}"] = safe(go.get(k)) if isinstance(go, dict) else None

    # ── detalle (último nodo)
    detalle = doc.get("detalle") or []
    d = detalle[-1] if detalle else {}
    for k in detalle_item_props:
        flat[f"detalle.{k}"] = safe(d.get(k)) if isinstance(d, dict) else None

    return flat


print("Función de normalización lista.")

Función de normalización lista.


In [7]:
# ── CONSTRUIR DATAFRAME ───────────────────────────────────────────────────────
if docs:
    rows = [flatten_doc(d) for d in docs]
    df   = pd.DataFrame(rows)
else:
    df = pd.DataFrame()

# Garantizar TODAS las columnas del schema (aunque no haya datos)
for col_name in ALL_COLS:
    if col_name not in df.columns:
        df[col_name] = None

# Reordenar exactamente como el schema
extra = [c for c in df.columns if c not in ALL_COLS]
df    = df[ALL_COLS + extra]

print(f"Shape final: {df.shape}")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head(3)

Shape final: (2787, 204)
Filas: 2787 | Columnas: 204


,_id,agente_id,agente_nombre,alias_contacto,anio_fin_atencion,anio_ingreso,anio_inicio_atencion,arranque_id,callid,campana_id,campana_nombre,chan_id,codigo_atencion,cola_id,cola_nombre,comunicacion_id,correlativo,dia_semana_fin_atencion_id,dia_semana_fin_atencion_nombre,dia_semana_ingreso_id,dia_semana_ingreso_nombre,dia_semana_inicio_atencion_id,dia_semana_inicio_atencion_nombre,estado_atencion,etiqueta_id,etiqueta_nombre,extension,fecha_fin,fecha_fin_atencion,fecha_fin_atencion_index,fecha_fin_comunicacion,fecha_ingreso_cola,fecha_ingreso_cola_index,fecha_inicio,fecha_inicio_atencion,fecha_inicio_atencion_data_comparacion,fecha_inicio_atencion_index,fecha_inicio_comunicacion,fecha_inicio_comunicacion_index,fecha_primer_registro,fecha_primera_respuesta_agente,fin_timbrado_marcador,finalizada_por,gestion_unica,hora_fin_atencion_index,hora_ingreso_cola_index,hora_inicio_atencion_index,hora_inicio_comunicacion_index,identificacion,inicio_timbrado_marcador,instancia,isMarcador,lote_id,lote_nombre,mes_fin_atencion,mes_ingreso,mes_inicio_atencion,nodo_id,nodo_nombre,nombre_sub_resultado,origen,proveedor_id,proveedor_nombre,publicidad_id,publicidad_identificador,publicidad_nombre,realizado_tipificacion_obligatoria,resultado_id,resultado_marcador_id,resultado_marcador_nombre,servicio_id,servicio_secundario,sip_code,sub_resultado_id,sub_tipo_marcador_id,sub_tipo_marcador_nombre,supera_umbral,tiempo,tiempo_atencion,tiempo_cola,tiempo_conversacion,tiempo_gestion,tiempo_ring,tiempo_ring_acumulado,tiempo_ring_agente,tiempo_timbrado_marcador,tipo_comunicacion_id,tipo_marcador_id,tipo_marcador_nombre,tipo_telefono_marcador,whatsapp_id,workflow_nombre,workflow_numero,field-1,field-2,contacto._id.date,contacto._id.timestamp,contacto.campana,contacto.carga_id,contacto.chan_id,contacto.contacto_id,contacto.contacto_nombre,contacto.contacto_validado,contacto.correos,contacto.departamento,contacto.dirección,contacto.distrito,contacto.facebooks,contacto.fecha_creacion,contacto.fecha_modificacion,contacto.from_marcador,contacto.id_campana,contacto.id_contacto,contacto.isDirty,contacto.lote_id,contacto.nombre,contacto.nombre_base,contacto.nro_de_documento,contacto.numeros,contacto.otros_distritos,contacto.provincia,contacto.telefonos,contacto.tipo_campaña,contacto.tipo_de_campaña,contacto.twitters,marcador_info.arranque_id,marcador_info.chan_id,marcador_info.code,marcador_info.dato1,marcador_info.dato10,marcador_info.dato11,marcador_info.dato12,marcador_info.dato13,marcador_info.dato14,marcador_info.dato15,marcador_info.dato16,marcador_info.dato17,marcador_info.dato18,marcador_info.dato19,marcador_info.dato2,marcador_info.dato20,marcador_info.dato3,marcador_info.dato4,marcador_info.dato5,marcador_info.dato6,marcador_info.dato7,marcador_info.dato8,marcador_info.dato9,marcador_info.lote_id,marcador_info.lote_nombre,marcador_info.nombre,marcador_info.numero,marcador_info.register_id,marcador_info.resultado_id,marcador_info.sub_tipo_marcador_id,marcador_info.sub_tipo_marcador_nombre,marcador_info.tipo_marcador_id,marcador_info.tipo_marcador_nombre,marcador_info.tipo_numero,marcador_info.field-1,gestiones.comentario,gestiones.fecha_index,gestiones.fecha_registro,gestiones.fecha_string,gestiones.hora_index,gestiones.nivel1,gestiones.nivel1_nombre,gestiones.nivel2,gestiones.nivel2_nombre,gestiones.nivel3,gestiones.nivel3_nombre,gestiones.nivel4,gestiones.nivel4_nombre,gestiones.resultado_id,gestiones.resultado_nombre,gestiones.tema_id,gestiones.tema_nombre,gestiones_old.comentario,gestiones_old.fecha_index,gestiones_old.fecha_registro,gestiones_old.fecha_string,gestiones_old.hora_index,gestiones_old.nivel1,gestiones_old.nivel1_nombre,gestiones_old.nivel2,gestiones_old.nivel2_nombre,gestiones_old.nivel3,gestiones_old.nivel3_nombre,gestiones_old.nivel4,gestiones_old.nivel4_nombre,gestiones_old.resultado_id,gestiones_old.resultado_nombre,gestiones_old.tema_id,gestiones_old.tema_nombre,detalle.comunicacion_id,detalle.fecha_fin,detalle.fecha_inicio,detalle.mens

In [8]:
# ── ANÁLISIS RÁPIDO DE NULOS ──────────────────────────────────────────────────
resumen = pd.DataFrame({
    "columna"    : df.columns,
    "con_dato"   : [df[c].notna().sum() for c in df.columns],
    "nulos"      : [df[c].isna().sum()  for c in df.columns],
})
resumen["%_nulo"]     = (resumen["nulos"]    / max(len(df), 1) * 100).round(1)
resumen["%_con_dato"] = (resumen["con_dato"] / max(len(df), 1) * 100).round(1)
resumen = resumen.sort_values("%_nulo").reset_index(drop=True)

print(f"Columnas con datos (>0 registros): {(resumen['con_dato'] > 0).sum()}")
print(f"Columnas 100% vacías             : {(resumen['%_nulo'] == 100).sum()}")
resumen

Columnas con datos (>0 registros): 145
Columnas 100% vacías             : 59


,columna,con_dato,nulos,%_nulo,%_con_dato
0,_id,2787,0,0.0,100.0
1,agente_id,2787,0,0.0,100.0
2,agente_nombre,2787,0,0.0,100.0
3,anio_ingreso,2787,0,0.0,100.0
4,codigo_atencion,2787,0,0.0,100.0
...,...,...,...,...,...
199,detalle.nodo_nombre,0,2787,100.0,0.0
200,detalle.respuesta_id,0,2787,100.0,0.0
201,detalle.respuesta_nombre,0,2787,100.0,0.0
202,detalle.tiempo,0,2787,100.0,0.0


In [9]:
# ── EXPORTAR A EXCEL ──────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:

    # Hoja 1: datos completos
    df.to_excel(writer, sheet_name="datos", index=False)

    # Hoja 2: resumen de nulos
    resumen.to_excel(writer, sheet_name="resumen_nulos", index=False)

    # Hoja 3: solo columnas con datos
    cols_con_dato = resumen[resumen["con_dato"] > 0]["columna"].tolist()
    df[cols_con_dato].to_excel(writer, sheet_name="solo_con_datos", index=False)

print(f"Excel guardado en: {OUTPUT_EXCEL}")
print(f"  Hoja 'datos'          : {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"  Hoja 'resumen_nulos'  : {len(resumen)} columnas analizadas")
print(f"  Hoja 'solo_con_datos' : {len(cols_con_dato)} columnas con al menos 1 dato")

Excel guardado en: C:\Users\USUARIO\Downloads\NTP_CODEX\extraccion_hoy.xlsx
  Hoja 'datos'          : 2787 filas x 204 columnas
  Hoja 'resumen_nulos'  : 204 columnas analizadas
  Hoja 'solo_con_datos' : 145 columnas con al menos 1 dato
